## **NoRegretsRVC**
Custom RVC voice model training environment built on Applio + Streamlit (32000 · HiFi-GAN · RMVPE).

- **Dataset Path**: `/kaggle/input/datasets/your-username/dataset`
- **Included Tabs**: samples, train, voice_blender, tts, models, settings
- **Tunnel & Auth**: Gradio + Cloudflare with auto-generated 24-byte WebUI password


## 1. Install Dependencies & Apply NoRegretsRVC Streamlit WebUI


In [ ]:
import base64
import ctypes.util
import hashlib
import os
import shutil
import subprocess
import sys
import urllib.request
from pathlib import Path
from IPython.display import clear_output

repo_dir = Path("/kaggle/working/program_ml")
repo_url = "https://github.com/IAHispano/Applio.git"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
if repo_dir.exists() and (repo_dir / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", repo_url, str(repo_dir)], check=True)

if shutil.which("fuser") is None or ctypes.util.find_library("portaudio") is None:
    subprocess.run(["apt", "update", "-y"], check=True)
    subprocess.run(["apt", "install", "-y", "portaudio19-dev", "psmisc"], check=True)

uv_bin = shutil.which("uv") or str(Path(sys.executable).with_name("uv"))
venv_python = Path("/kaggle/.venv/bin/python")
if not venv_python.exists():
    subprocess.run([uv_bin, "venv", "/kaggle/.venv", "--python", "3.12", "--seed"], check=True)
py_exec = str(venv_python)
subprocess.run([uv_bin, "pip", "install", "-q", "-r", str(repo_dir / "requirements.txt"), "--extra-index-url", "https://download.pytorch.org/whl/cu128", "--index-strategy", "unsafe-best-match", "--python", py_exec], check=True)
subprocess.run([uv_bin, "pip", "install", "-q", "streamlit>=1.40.0", "matplotlib-inline", "websockets", "--python", py_exec], check=True)
subprocess.run([uv_bin, "pip", "install", "-q", "onnxruntime-gpu", "--python", py_exec] if True else [uv_bin, "pip", "install", "-q", "onnxruntime-gpu", "--system"], check=False)
subprocess.run([py_exec, "-c", "import onnxruntime as ort; print('ONNX Providers:', ort.get_available_providers())"], check=False)

# Reset previously patched config so re-runs stay clean
if (repo_dir / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo_dir), "checkout", "--", "assets/config_template.json"], check=False)

# Enforce NoRegretsRVC engine defaults (fp32 precision, Discord presence off)
import json as _cfg_json
for _cfg_rel in ("assets/config_template.json", "assets/config.json"):
    _cfg_p = repo_dir / _cfg_rel
    if _cfg_p.is_file():
        try:
            _cfg_d = _cfg_json.loads(_cfg_p.read_text(encoding="utf-8"))
            _cfg_d["precision"] = "fp32"
            _cfg_d["discord_presence"] = False
            _cfg_p.write_text(_cfg_json.dumps(_cfg_d, indent=4, ensure_ascii=False), encoding="utf-8")
        except Exception:
            pass

# Write NoRegretsRVC Streamlit WebUI, headless credentials, and locked crimson theme
for _st_dir in (repo_dir / ".streamlit", Path.home() / ".streamlit"):
    _st_dir.mkdir(parents=True, exist_ok=True)
    (_st_dir / "credentials.toml").write_text('[general]\nemail = ""\n', encoding="utf-8")
(repo_dir / ".streamlit" / "config.toml").write_bytes(base64.b64decode("W3RoZW1lXQpiYXNlID0gImRhcmsiCnByaW1hcnlDb2xvciA9ICIjRTUxOTIzIgpiYWNrZ3JvdW5kQ29sb3IgPSAiIzBEMDQwNSIKc2Vjb25kYXJ5QmFja2dyb3VuZENvbG9yID0gIiMxNjA3MDkiCnRleHRDb2xvciA9ICIjRkRGN0Y3Igpmb250ID0gInNhbnMgc2VyaWYiCgpbc2VydmVyXQpoZWFkbGVzcyA9IHRydWUKZW5hYmxlQ09SUyA9IGZhbHNlCmVuYWJsZVhzcmZQcm90ZWN0aW9uID0gZmFsc2UKbWF4VXBsb2FkU2l6ZSA9IDIwNDgKClticm93c2VyXQpnYXRoZXJVc2FnZVN0YXRzID0gZmFsc2UK"))
(repo_dir / "streamlit_app.py").write_bytes(base64.b64decode("..."))
(repo_dir / "logs").mkdir(parents=True, exist_ok=True)
(repo_dir / "noregrets_turbo.py").write_bytes(base64.b64decode("ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGF0ZXhpdApmcm9tIGNvbmN1cnJlbnQuZnV0dXJlcyBpbXBvcnQgRnV0dXJlLCBUaHJlYWRQb29sRXhlY3V0b3IKaW1wb3J0IGluc3BlY3QKaW1wb3J0IGlvCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHRocmVhZGluZwpmcm9tIHR5cGluZyBpbXBvcnQgQW55CgpfQVBQTElFRCA9IEZhbHNlCl9DQUNIRV9MT0NLID0gdGhyZWFkaW5nLkxvY2soKQpfQVJSQVlfQ0FDSEU6IGRpY3Rbc3RyLCBBbnldID0ge30KX0NBQ0hFX0JZVEVTID0gMApfQ0FDSEVfQ0hFQ0tTID0gMApfUEVORElOR19TQVZFUzogZGljdFtzdHIsIEZ1dHVyZVtOb25lXV0gPSB7fQpfU0FWRV9MT0NLID0gdGhyZWFkaW5nLkxvY2soKQpfU0FWRV9QT09MOiBUaHJlYWRQb29sRXhlY3V0b3IgfCBOb25lID0gTm9uZQoKX0RBVEFTRVRfU1VCRElSUyA9ICgic2xpY2VkX2F1ZGlvcyIsICJleHRyYWN0ZWQiLCAiZjAiLCAiZjBfdm9pY2VkIikKX01BWF9TSU5HTEVfRklMRV9CWVRFUyA9IDE2ICogMTAyNCAqIDEwMjQKX1BFUl9QUk9DRVNTX1JBTV9DQVBfQllURVMgPSAyICogMTAyNCAqIDEwMjQgKiAxMDI0Cl9NSU5fU1lTVEVNX0ZSRUVfUkFNX0JZVEVTID0gOCAqIDEwMjQgKiAxMDI0ICogMTAyNAoKCmRlZiBfcmVhZF9tZW1fYXZhaWxhYmxlX2J5dGVzKCkgLT4gaW50OgogICAgdHJ5OgogICAgICAgIHdpdGggb3BlbigiL3Byb2MvbWVtaW5mbyIsICJyIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZmg6CiAgICAgICAgICAgIGZvciBsaW5lIGluIGZoOgogICAgICAgICAgICAgICAgaWYgbGluZS5zdGFydHN3aXRoKCJNZW1BdmFpbGFibGU6Iik6CiAgICAgICAgICAgICAgICAgICAga2IgPSBpbnQobGluZS5zcGxpdCgpWzFdKQogICAgICAgICAgICAgICAgICAgIHJldHVybiBrYiAqIDEwMjQKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcGFzcwogICAgcmV0dXJuIDAKCgpkZWYgX2hhc19zYWZlX3JhbShpbmNvbWluZ19ieXRlczogaW50KSAtPiBib29sOgogICAgZ2xvYmFsIF9DQUNIRV9DSEVDS1MKICAgIGlmIF9DQUNIRV9CWVRFUyArIGluY29taW5nX2J5dGVzID4gX1BFUl9QUk9DRVNTX1JBTV9DQVBfQllURVM6CiAgICAgICAgcmV0dXJuIEZhbHNlCiAgICBfQ0FDSEVfQ0hFQ0tTICs9IDEKICAgIGlmIF9DQUNIRV9DSEVDS1MgPT0gMSBvciBfQ0FDSEVfQ0hFQ0tTICUgMjQgPT0gMDoKICAgICAgICBhdmFpbCA9IF9yZWFkX21lbV9hdmFpbGFibGVfYnl0ZXMoKQogICAgICAgIGlmIGF2YWlsID4gMCBhbmQgYXZhaWwgLSBpbmNvbWluZ19ieXRlcyA8IF9NSU5fU1lTVEVNX0ZSRUVfUkFNX0JZVEVTOgogICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgIHJldHVybiBUcnVlCgoKZGVmIF9pc19jYWNoZWFibGVfZGF0YXNldF9wYXRoKHBhdGg6IEFueSkgLT4gc3RyIHwgTm9uZToKICAgIGlmIG5vdCBpc2luc3RhbmNlKHBhdGgsIChzdHIsIG9zLlBhdGhMaWtlKSk6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIHJhdyA9IG9zLmZzcGF0aChwYXRoKQogICAgbm9ybSA9IHJhdy5yZXBsYWNlKCJcXCIsICIvIikKICAgIGlmICIvbG9ncy8iIG5vdCBpbiBub3JtIGFuZCBub3Qgbm9ybS5zdGFydHN3aXRoKCJsb2dzLyIpOgogICAgICAgIHJldHVybiBOb25lCiAgICBpZiBub3QgYW55KGYiL3twYXJ0fS8iIGluIG5vcm0gb3Igbm9ybS5zdGFydHN3aXRoKGYie3BhcnR9LyIpIGZvciBwYXJ0IGluIF9EQVRBU0VUX1NVQkRJUlMpOgogICAgICAgIHJldHVybiBOb25lCiAgICB0cnk6CiAgICAgICAgaWYgb3MucGF0aC5nZXRzaXplKHJhdykgPiBfTUFYX1NJTkdMRV9GSUxFX0JZVEVTOgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgZXhjZXB0IE9TRXJyb3I6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIHJldHVybiByYXcKCgpkZWYgX2FwcGx5X2N1ZGFfdHVuaW5nKCkgLT4gTm9uZToKICAgIGltcG9ydCB0b3JjaAoKICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgdG9yY2guYmFja2VuZHMuY3Vkbm4uZW5hYmxlZCA9IFRydWUKICAgICAgICB0b3JjaC5iYWNrZW5kcy5jdWRubi5iZW5jaG1hcmsgPSBUcnVlCgoKZGVmIF9wYXRjaF9kYXRhbG9hZGVyKCkgLT4gTm9uZToKICAgIGltcG9ydCB0b3JjaC51dGlscy5kYXRhCgogICAgb3JpZ19pbml0ID0gdG9yY2gudXRpbHMuZGF0YS5EYXRhTG9hZGVyLl9faW5pdF9fCiAgICBpZiBnZXRhdHRyKG9yaWdfaW5pdCwgIl9ub3JlZ3JldHNfcGF0Y2hlZCIsIEZhbHNlKToKICAgICAgICByZXR1cm4KCiAgICB3b3JsZF9zaXplID0gbWF4KDEsIGludChvcy5lbnZpcm9uLmdldCgiV09STERfU0laRSIsICIxIikpKQogICAgY3B1X2NhcCA9IG1heCgxLCBtaW4oNCwgKG9zLmNwdV9jb3VudCgpIG9yIDQpIC8vIHdvcmxkX3NpemUpKQoKICAgIGRlZiBmYXN0X2luaXQoc2VsZjogQW55LCAqYXJnczogQW55LCAqKmt3YXJnczogQW55KSAtPiBBbnk6CiAgICAgICAgbnVtX3dvcmtlcnMgPSBrd2FyZ3MuZ2V0KCJudW1fd29ya2VycyIsIDApCiAgICAgICAgaWYgIm51bV93b3JrZXJzIiBub3QgaW4ga3dhcmdzIGFuZCBsZW4oYXJncykgPj0gNjoKICAgICAgICAgICAgbnVtX3dvcmtlcnMgPSBhcmdzWzVdCiAgICAgICAgaWYgaXNpbnN0YW5jZShudW1fd29ya2VycywgaW50KSBhbmQgbnVtX3dvcmtlcnMgPiAwOgogICAgICAgICAgICBpZiAibnVtX3dvcmtlcnMiIGluIGt3YXJncyBhbmQgbnVtX3dvcmtlcnMgPiBjcHVfY2FwOgogICAgICAgICAgICAgICAga3dhcmdzWyJudW1fd29ya2VycyJdID0gY3B1X2NhcAogICAgICAgICAgICBrd2FyZ3Muc2V0ZGVmYXVsdCgicGluX21lbW9yeSIsIFRydWUpCiAgICAgICAgICAgIGt3YXJncy5zZXRkZWZhdWx0KCJwZXJzaXN0ZW50X3dvcmtlcnMiLCBUcnVlKQogICAgICAgICAgICBrd2FyZ3Muc2V0ZGVmYXVsdCgicHJlZmV0Y2hfZmFjdG9yIiwgMikKICAgICAgICB0cnk6CiAgICAgICAgICAgIHJldHVybiBvcmlnX2luaXQoc2VsZiwgKmFyZ3MsICoqa3dhcmdzKQogICAgICAgIGV4Y2VwdCAoVHlwZUVycm9yLCBWYWx1ZUVycm9yKToKICAgICAgICAgICAga3dhcmdzLnBvcCgicGVyc2lzdGVudF93b3JrZXJzIiwgTm9uZSkKICAgICAgICAgICAga3dhcmdzLnBvcCgicHJlZmV0Y2hfZmFjdG9yIiwgTm9uZSkKICAgICAgICAgICAga3dhcmdzLnBvcCgicGluX21lbW9yeSIsIE5vbmUpCiAgICAgICAgICAgIHJldHVybiBvcmlnX2luaXQoc2VsZiwgKmFyZ3MsICoqa3dhcmdzKQoKICAgIGZhc3RfaW5pdC5fbm9yZWdyZXRzX3BhdGNoZWQgPSBUcnVlICAjIHR5cGU6IGlnbm9yZVthdHRyLWRlZmluZWRdCiAgICB0b3JjaC51dGlscy5kYXRhLkRhdGFMb2FkZXIuX19pbml0X18gPSBmYXN0X2luaXQgICMgdHlwZTogaWdub3JlW21ldGhvZC1hc3NpZ25dCgoKZGVmIF9wYXRjaF9vcHRpbWl6ZXIoKSAtPiBOb25lOgogICAgaW1wb3J0IHRvcmNoCgogICAgb3JpZ19hZGFtd19pbml0ID0gdG9yY2gub3B0aW0uQWRhbVcuX19pbml0X18KICAgIGlmIG5vdCBnZXRhdHRyKG9yaWdfYWRhbXdfaW5pdCwgIl9ub3JlZ3JldHNfcGF0Y2hlZCIsIEZhbHNlKToKICAgICAgICBzdXBwb3J0c19mb3JlYWNoID0gImZvcmVhY2giIGluIGluc3BlY3Quc2lnbmF0dXJlKG9yaWdfYWRhbXdfaW5pdCkucGFyYW1ldGVycwogICAgICAgIHdvcmxkX3NpemUgPSBtYXgoMSwgaW50KG9zLmVudmlyb24uZ2V0KCJXT1JMRF9TSVpFIiwgIjEiKSkpCiAgICAgICAgdHJ5OgogICAgICAgICAgICBkZHBfbHJfc2NhbGUgPSBtYXgoMS4wLCBtaW4oMS41LCBmbG9hdChvcy5lbnZpcm9uLmdldCgiTlJfRERQX0xSX1NDQUxFIiwgIjEuMCIpKSkpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgZGRwX2xyX3NjYWxlID0gMS4wCgogICAgICAgIGRlZiBmYXN0X2FkYW13X2luaXQoc2VsZjogQW55LCBwYXJhbXM6IEFueSwgKmFyZ3M6IEFueSwgKiprd2FyZ3M6IEFueSkgLT4gQW55OgogICAgICAgICAgICBfYXBwbHlfY3VkYV90dW5pbmcoKQogICAgICAgICAgICBpZiB3b3JsZF9zaXplID49IDIgYW5kIGRkcF9scl9zY2FsZSA+IDEuMDAxIGFuZCAibHIiIGluIGt3YXJncyBhbmQgaXNpbnN0YW5jZShrd2FyZ3NbImxyIl0sIChpbnQsIGZsb2F0KSk6CiAgICAgICAgICAgICAgICBrd2FyZ3NbImxyIl0gPSBmbG9hdChrd2FyZ3NbImxyIl0pICogZGRwX2xyX3NjYWxlCiAgICAgICAgICAgIGlmIHN1cHBvcnRzX2ZvcmVhY2ggYW5kIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgYW5kICJmb3JlYWNoIiBub3QgaW4ga3dhcmdzIGFuZCBub3Qga3dhcmdzLmdldCgiZnVzZWQiLCBGYWxzZSk6CiAgICAgICAgICAgICAgICBrd2FyZ3NbImZvcmVhY2giXSA9IFRydWUKICAgICAgICAgICAgcmV0dXJuIG9yaWdfYWRhbXdfaW5pdChzZWxmLCBwYXJhbXMsICphcmdzLCAqKmt3YXJncykKCiAgICAgICAgZmFzdF9hZGFtd19pbml0Ll9ub3JlZ3JldHNfcGF0Y2hlZCA9IFRydWUgICMgdHlwZTogaWdub3JlW2F0dHItZGVmaW5lZF0KICAgICAgICB0b3JjaC5vcHRpbS5BZGFtVy5fX2luaXRfXyA9IGZhc3RfYWRhbXdfaW5pdCAgIyB0eXBlOiBpZ25vcmVbbWV0aG9kLWFzc2lnbl0KCiAgICBvcmlnX3plcm9fZ3JhZCA9IHRvcmNoLm9wdGltLk9wdGltaXplci56ZXJvX2dyYWQKICAgIGlmIG5vdCBnZXRhdHRyKG9yaWdfemVyb19ncmFkLCAiX25vcmVncmV0c19wYXRjaGVkIiwgRmFsc2UpOgogICAgICAgIGRlZiBmYXN0X3plcm9fZ3JhZChzZWxmOiBBbnksIHNldF90b19ub25lOiBib29sID0gVHJ1ZSkgLT4gQW55OgogICAgICAgICAgICByZXR1cm4gb3JpZ196ZXJvX2dyYWQoc2VsZiwgc2V0X3RvX25vbmU9c2V0X3RvX25vbmUpCgogICAgICAgIGZhc3RfemVyb19ncmFkLl9ub3JlZ3JldHNfcGF0Y2hlZCA9IFRydWUgICMgdHlwZTogaWdub3JlW2F0dHItZGVmaW5lZF0KICAgICAgICB0b3JjaC5vcHRpbS5PcHRpbWl6ZXIuemVyb19ncmFkID0gZmFzdF96ZXJvX2dyYWQgICMgdHlwZTogaWdub3JlW21ldGhvZC1hc3NpZ25dCgoKZGVmIF9wYXRjaF9kZHAoKSAtPiBOb25lOgogICAgaW1wb3J0IHRvcmNoLmRpc3RyaWJ1dGVkIGFzIGRpc3QKICAgIGltcG9ydCB0b3JjaC5ubi5wYXJhbGxlbAoKICAgIGRkcF9jbHMgPSB0b3JjaC5ubi5wYXJhbGxlbC5EaXN0cmlidXRlZERhdGFQYXJhbGxlbAogICAgb3JpZ19kZHBfaW5pdCA9IGRkcF9jbHMuX19pbml0X18KICAgIGlmIG5vdCBnZXRhdHRyKG9yaWdfZGRwX2luaXQsICJfbm9yZWdyZXRzX3BhdGNoZWQiLCBGYWxzZSk6CiAgICAgICAgZGVmIGZhc3RfZGRwX2luaXQoc2VsZjogQW55LCAqYXJnczogQW55LCAqKmt3YXJnczogQW55KSAtPiBBbnk6CiAgICAgICAgICAgIF9hcHBseV9jdWRhX3R1bmluZygpCiAgICAgICAgICAgIGt3YXJncy5zZXRkZWZhdWx0KCJncmFkaWVudF9hc19idWNrZXRfdmlldyIsIFRydWUpCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIHJldHVybiBvcmlnX2RkcF9pbml0KHNlbGYsICphcmdzLCAqKmt3YXJncykKICAgICAgICAgICAgZXhjZXB0IFR5cGVFcnJvcjoKICAgICAgICAgICAgICAgIGt3YXJncy5wb3AoImdyYWRpZW50X2FzX2J1Y2tldF92aWV3IiwgTm9uZSkKICAgICAgICAgICAgICAgIHJldHVybiBvcmlnX2RkcF9pbml0KHNlbGYsICphcmdzLCAqKmt3YXJncykKCiAgICAgICAgZmFzdF9kZHBfaW5pdC5fbm9yZWdyZXRzX3BhdGNoZWQgPSBUcnVlICAjIHR5cGU6IGlnbm9yZVthdHRyLWRlZmluZWRdCiAgICAgICAgZGRwX2Nscy5fX2luaXRfXyA9IGZhc3RfZGRwX2luaXQgICMgdHlwZTogaWdub3JlW21ldGhvZC1hc3NpZ25dCgogICAgb3JpZ19kZXN0cm95ID0gZ2V0YXR0cihkaXN0LCAiZGVzdHJveV9wcm9jZXNzX2dyb3VwIiwgTm9uZSkKICAgIGlmIG9yaWdfZGVzdHJveSBpcyBub3QgTm9uZSBhbmQgbm90IGdldGF0dHIob3JpZ19kZXN0cm95LCAiX25vcmVncmV0c19wYXRjaGVkIiwgRmFsc2UpOgogICAgICAgIGRlZiBzYWZlX2Rlc3Ryb3lfcHJvY2Vzc19ncm91cCgqYXJnczogQW55LCAqKmt3YXJnczogQW55KSAtPiBBbnk6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIGZsdXNoX3NhdmVzKCkKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgIHBhc3MKICAgICAgICAgICAgcmV0dXJuIG9yaWdfZGVzdHJveSgqYXJncywgKiprd2FyZ3MpCgogICAgICAgIHNhZmVfZGVzdHJveV9wcm9jZXNzX2dyb3VwLl9ub3JlZ3JldHNfcGF0Y2hlZCA9IFRydWUgICMgdHlwZTogaWdub3JlW2F0dHItZGVmaW5lZF0KICAgICAgICBkaXN0LmRlc3Ryb3lfcHJvY2Vzc19ncm91cCA9IHNhZmVfZGVzdHJveV9wcm9jZXNzX2dyb3VwICAjIHR5cGU6IGlnbm9yZVthc3NpZ25tZW50XQoKCmRlZiBfcGF0Y2hfdGVuc29yYm9hcmRfaW1hZ2VzKCkgLT4gTm9uZToKICAgIHRyeToKICAgICAgICBmcm9tIHRvcmNoLnV0aWxzLnRlbnNvcmJvYXJkIGltcG9ydCBTdW1tYXJ5V3JpdGVyCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHJldHVybgoKICAgIGRlZiBfc2tpcF9pbWFnZSgqYXJnczogQW55LCAqKmt3YXJnczogQW55KSAtPiBOb25lOgogICAgICAgIHJldHVybiBOb25lCgogICAgZm9yIG1ldGhvZCBpbiAoImFkZF9pbWFnZSIsICJhZGRfaW1hZ2VzIiwgImFkZF9maWd1cmUiKToKICAgICAgICBpZiBoYXNhdHRyKFN1bW1hcnlXcml0ZXIsIG1ldGhvZCk6CiAgICAgICAgICAgIHNldGF0dHIoU3VtbWFyeVdyaXRlciwgbWV0aG9kLCBfc2tpcF9pbWFnZSkKCgpkZWYgX3BhdGNoX3JhbV9jYWNoZSgpIC0+IE5vbmU6CiAgICBpbXBvcnQgbnVtcHkgYXMgbnAKCiAgICBvcmlnX25wX2xvYWQgPSBucC5sb2FkCiAgICBpZiBub3QgZ2V0YXR0cihvcmlnX25wX2xvYWQsICJfbm9yZWdyZXRzX3BhdGNoZWQiLCBGYWxzZSk6CiAgICAgICAgZGVmIGZhc3RfbnBfbG9hZChmaWxlOiBBbnksICphcmdzOiBBbnksICoqa3dhcmdzOiBBbnkpIC0+IEFueToKICAgICAgICAgICAgZ2xvYmFsIF9DQUNIRV9CWVRFUwogICAgICAgICAgICBpZiBhcmdzIG9yIGt3YXJnczoKICAgICAgICAgICAgICAgIHJldHVybiBvcmlnX25wX2xvYWQoZmlsZSwgKmFyZ3MsICoqa3dhcmdzKQogICAgICAgICAgICBrZXkgPSBfaXNfY2FjaGVhYmxlX2RhdGFzZXRfcGF0aChmaWxlKQogICAgICAgICAgICBpZiBrZXkgaXMgTm9uZToKICAgICAgICAgICAgICAgIHJldHVybiBvcmlnX25wX2xvYWQoZmlsZSkKICAgICAgICAgICAgY2FjaGVfa2V5ID0gZiJucDo6e2tleX0iCiAgICAgICAgICAgIHdpdGggX0NBQ0hFX0xPQ0s6CiAgICAgICAgICAgICAgICBjYWNoZWQgPSBfQVJSQVlfQ0FDSEUuZ2V0KGNhY2hlX2tleSkKICAgICAgICAgICAgaWYgY2FjaGVkIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgcmV0dXJuIGNhY2hlZC5jb3B5KCkKICAgICAgICAgICAgYXJyID0gb3JpZ19ucF9sb2FkKGZpbGUpCiAgICAgICAgICAgIG5ieXRlcyA9IGludChnZXRhdHRyKGFyciwgIm5ieXRlcyIsIDApKQogICAgICAgICAgICBpZiBuYnl0ZXMgPiAwIGFuZCBpc2luc3RhbmNlKGFyciwgbnAubmRhcnJheSk6CiAgICAgICAgICAgICAgICB3aXRoIF9DQUNIRV9MT0NLOgogICAgICAgICAgICAgICAgICAgIGlmIGNhY2hlX2tleSBub3QgaW4gX0FSUkFZX0NBQ0hFIGFuZCBfaGFzX3NhZmVfcmFtKG5ieXRlcyk6CiAgICAgICAgICAgICAgICAgICAgICAgIF9BUlJBWV9DQUNIRVtjYWNoZV9rZXldID0gYXJyLmNvcHkoKQogICAgICAgICAgICAgICAgICAgICAgICBfQ0FDSEVfQllURVMgKz0gbmJ5dGVzCiAgICAgICAgICAgIHJldHVybiBhcnIKCiAgICAgICAgZmFzdF9ucF9sb2FkLl9ub3JlZ3JldHNfcGF0Y2hlZCA9IFRydWUgICMgdHlwZTogaWdub3JlW2F0dHItZGVmaW5lZF0KICAgICAgICBucC5sb2FkID0gZmFzdF9ucF9sb2FkICAjIHR5cGU6IGlnbm9yZVthc3NpZ25tZW50XQoKICAgIHRyeToKICAgICAgICBpbXBvcnQgc2NpcHkuaW8ud2F2ZmlsZSBhcyB3YXZmaWxlCgogICAgICAgIG9yaWdfd2F2X3JlYWQgPSB3YXZmaWxlLnJlYWQKICAgICAgICBpZiBub3QgZ2V0YXR0cihvcmlnX3dhdl9yZWFkLCAiX25vcmVncmV0c19wYXRjaGVkIiwgRmFsc2UpOgogICAgICAgICAgICBkZWYgZmFzdF93YXZfcmVhZChmaWxlbmFtZTogQW55LCAqYXJnczogQW55LCAqKmt3YXJnczogQW55KSAtPiBBbnk6CiAgICAgICAgICAgICAgICBnbG9iYWwgX0NBQ0hFX0JZVEVTCiAgICAgICAgICAgICAgICBpZiBhcmdzIG9yIGt3YXJnczoKICAgICAgICAgICAgICAgICAgICByZXR1cm4gb3JpZ193YXZfcmVhZChmaWxlbmFtZSwgKmFyZ3MsICoqa3dhcmdzKQogICAgICAgICAgICAgICAga2V5ID0gX2lzX2NhY2hlYWJsZV9kYXRhc2V0X3BhdGgoZmlsZW5hbWUpCiAgICAgICAgICAgICAgICBpZiBrZXkgaXMgTm9uZToKICAgICAgICAgICAgICAgICAgICByZXR1cm4gb3JpZ193YXZfcmVhZChmaWxlbmFtZSkKICAgICAgICAgICAgICAgIGNhY2hlX2tleSA9IGYid2F2Ojp7a2V5fSIKICAgICAgICAgICAgICAgIHdpdGggX0NBQ0hFX0xPQ0s6CiAgICAgICAgICAgICAgICAgICAgY2FjaGVkID0gX0FSUkFZX0NBQ0hFLmdldChjYWNoZV9rZXkpCiAgICAgICAgICAgICAgICBpZiBjYWNoZWQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgc3IsIGRhdGEgPSBjYWNoZWQKICAgICAgICAgICAgICAgICAgICByZXR1cm4gc3IsIGRhdGEuY29weSgpCiAgICAgICAgICAgICAgICBzciwgZGF0YSA9IG9yaWdfd2F2X3JlYWQoZmlsZW5hbWUpCiAgICAgICAgICAgICAgICBuYnl0ZXMgPSBpbnQoZ2V0YXR0cihkYXRhLCAibmJ5dGVzIiwgMCkpCiAgICAgICAgICAgICAgICBpZiBuYnl0ZXMgPiAwOgogICAgICAgICAgICAgICAgICAgIHdpdGggX0NBQ0hFX0xPQ0s6CiAgICAgICAgICAgICAgICAgICAgICAgIGlmIGNhY2hlX2tleSBub3QgaW4gX0FSUkFZX0NBQ0hFIGFuZCBfaGFzX3NhZmVfcmFtKG5ieXRlcyk6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICBfQVJSQVlfQ0FDSEVbY2FjaGVfa2V5XSA9IChzciwgZGF0YS5jb3B5KCkpCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBfQ0FDSEVfQllURVMgKz0gbmJ5dGVzCiAgICAgICAgICAgICAgICByZXR1cm4gc3IsIGRhdGEKCiAgICAgICAgICAgIGZhc3Rfd2F2X3JlYWQuX25vcmVncmV0c19wYXRjaGVkID0gVHJ1ZSAgIyB0eXBlOiBpZ25vcmVbYXR0ci1kZWZpbmVkXQogICAgICAgICAgICB3YXZmaWxlLnJlYWQgPSBmYXN0X3dhdl9yZWFkICAjIHR5cGU6IGlnbm9yZVthc3NpZ25tZW50XQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICBwYXNzCgogICAgdHJ5OgogICAgICAgIGltcG9ydCBzb3VuZGZpbGUgYXMgc2YKCiAgICAgICAgb3JpZ19zZl9yZWFkID0gc2YucmVhZAogICAgICAgIGlmIG5vdCBnZXRhdHRyKG9yaWdfc2ZfcmVhZCwgIl9ub3JlZ3JldHNfcGF0Y2hlZCIsIEZhbHNlKToKICAgICAgICAgICAgZGVmIGZhc3Rfc2ZfcmVhZChmaWxlOiBBbnksICphcmdzOiBBbnksICoqa3dhcmdzOiBBbnkpIC0+IEFueToKICAgICAgICAgICAgICAgIGdsb2JhbCBfQ0FDSEVfQllURVMKICAgICAgICAgICAgICAgIGlmIGFyZ3Mgb3Iga3dhcmdzOgogICAgICAgICAgICAgICAgICAgIHJldHVybiBvcmlnX3NmX3JlYWQoZmlsZSwgKmFyZ3MsICoqa3dhcmdzKQogICAgICAgICAgICAgICAga2V5ID0gX2lzX2NhY2hlYWJsZV9kYXRhc2V0X3BhdGgoZmlsZSkKICAgICAgICAgICAgICAgIGlmIGtleSBpcyBOb25lOgogICAgICAgICAgICAgICAgICAgIHJldHVybiBvcmlnX3NmX3JlYWQoZmlsZSkKICAgICAgICAgICAgICAgIGNhY2hlX2tleSA9IGYic2Y6OntrZXl9IgogICAgICAgICAgICAgICAgd2l0aCBfQ0FDSEVfTE9DSzoKICAgICAgICAgICAgICAgICAgICBjYWNoZWQgPSBfQVJSQVlfQ0FDSEUuZ2V0KGNhY2hlX2tleSkKICAgICAgICAgICAgICAgIGlmIGNhY2hlZCBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgICAgICBkYXRhLCBzciA9IGNhY2hlZAogICAgICAgICAgICAgICAgICAgIHJldHVybiBkYXRhLmNvcHkoKSwgc3IKICAgICAgICAgICAgICAgIGRhdGEsIHNyID0gb3JpZ19zZl9yZWFkKGZpbGUpCiAgICAgICAgICAgICAgICBuYnl0ZXMgPSBpbnQoZ2V0YXR0cihkYXRhLCAibmJ5dGVzIiwgMCkpCiAgICAgICAgICAgICAgICBpZiBuYnl0ZXMgPiAwOgogICAgICAgICAgICAgICAgICAgIHdpdGggX0NBQ0hFX0xPQ0s6CiAgICAgICAgICAgICAgICAgICAgICAgIGlmIGNhY2hlX2tleSBub3QgaW4gX0FSUkFZX0NBQ0hFIGFuZCBfaGFzX3NhZmVfcmFtKG5ieXRlcyk6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICBfQVJSQVlfQ0FDSEVbY2FjaGVfa2V5XSA9IChkYXRhLmNvcHkoKSwgc3IpCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBfQ0FDSEVfQllURVMgKz0gbmJ5dGVzCiAgICAgICAgICAgICAgICByZXR1cm4gZGF0YSwgc3IKCiAgICAgICAgICAgIGZhc3Rfc2ZfcmVhZC5fbm9yZWdyZXRzX3BhdGNoZWQgPSBUcnVlICAjIHR5cGU6IGlnbm9yZVthdHRyLWRlZmluZWRdCiAgICAgICAgICAgIHNmLnJlYWQgPSBmYXN0X3NmX3JlYWQgICMgdHlwZTogaWdub3JlW2Fzc2lnbm1lbnRdCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHBhc3MKCgpkZWYgX3dyaXRlX2F0b21pYyh0YXJnZXQ6IFBhdGgsIHBheWxvYWQ6IGJ5dGVzKSAtPiBOb25lOgogICAgdGFyZ2V0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB0bXAgPSB0YXJnZXQud2l0aF9uYW1lKGYiLnt0YXJnZXQubmFtZX0udG1wIikKICAgIHRyeToKICAgICAgICB3aXRoIG9wZW4odG1wLCAid2IiKSBhcyBmaDoKICAgICAgICAgICAgZmgud3JpdGUocGF5bG9hZCkKICAgICAgICBvcy5yZXBsYWNlKHRtcCwgdGFyZ2V0KQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICB0cnk6CiAgICAgICAgICAgIGlmIHRtcC5leGlzdHMoKToKICAgICAgICAgICAgICAgIHRtcC51bmxpbmsoKQogICAgICAgIGV4Y2VwdCBPU0Vycm9yOgogICAgICAgICAgICBwYXNzCiAgICAgICAgcmFpc2UKCgpkZWYgZmx1c2hfc2F2ZXMoKSAtPiBOb25lOgogICAgd2l0aCBfU0FWRV9MT0NLOgogICAgICAgIHBlbmRpbmcgPSBsaXN0KF9QRU5ESU5HX1NBVkVTLml0ZW1zKCkpCiAgICBmb3Iga2V5LCBmdXQgaW4gcGVuZGluZzoKICAgICAgICB0cnk6CiAgICAgICAgICAgIGZ1dC5yZXN1bHQoKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHBhc3MKICAgICAgICB3aXRoIF9TQVZFX0xPQ0s6CiAgICAgICAgICAgIF9QRU5ESU5HX1NBVkVTLnBvcChrZXksIE5vbmUpCgoKZGVmIF9zaHV0ZG93bl9zYXZlX3Bvb2woKSAtPiBOb25lOgogICAgZmx1c2hfc2F2ZXMoKQogICAgaWYgX1NBVkVfUE9PTCBpcyBub3QgTm9uZToKICAgICAgICBfU0FWRV9QT09MLnNodXRkb3duKHdhaXQ9VHJ1ZSkKCgpkZWYgX3BhdGNoX2FzeW5jX3NhdmUoKSAtPiBOb25lOgogICAgZ2xvYmFsIF9TQVZFX1BPT0wKICAgIGltcG9ydCB0b3JjaAoKICAgIG9yaWdfc2F2ZSA9IHRvcmNoLnNhdmUKICAgIG9yaWdfbG9hZCA9IHRvcmNoLmxvYWQKICAgIGlmIGdldGF0dHIob3JpZ19zYXZlLCAiX25vcmVncmV0c19wYXRjaGVkIiwgRmFsc2UpOgogICAgICAgIHJldHVybgoKICAgIF9TQVZFX1BPT0wgPSBUaHJlYWRQb29sRXhlY3V0b3IobWF4X3dvcmtlcnM9MSwgdGhyZWFkX25hbWVfcHJlZml4PSJuci1ja3B0IikKICAgIGF0ZXhpdC5yZWdpc3Rlcihfc2h1dGRvd25fc2F2ZV9wb29sKQoKICAgIGRlZiBmYXN0X3NhdmUob2JqOiBBbnksIGY6IEFueSwgKmFyZ3M6IEFueSwgKiprd2FyZ3M6IEFueSkgLT4gQW55OgogICAgICAgIGlmICgKICAgICAgICAgICAgX1NBVkVfUE9PTCBpcyBub3QgTm9uZQogICAgICAgICAgICBhbmQgaXNpbnN0YW5jZShmLCAoc3RyLCBvcy5QYXRoTGlrZSkpCiAgICAgICAgICAgIGFuZCBub3QgYXJncwogICAgICAgICAgICBhbmQgbm90IGt3YXJncwogICAgICAgICk6CiAgICAgICAgICAgIHJhdyA9IG9zLmZzcGF0aChmKQogICAgICAgICAgICBub3JtID0gcmF3LnJlcGxhY2UoIlxcIiwgIi8iKQogICAgICAgICAgICBpZiBub3JtLmVuZHN3aXRoKCIucHRoIikgYW5kICgiL2xvZ3MvIiBpbiBub3JtIG9yIG5vcm0uc3RhcnRzd2l0aCgibG9ncy8iKSk6CiAgICAgICAgICAgICAgICB0YXJnZXQgPSBQYXRoKHJhdykucmVzb2x2ZSgpCiAgICAgICAgICAgICAgICBmbmFtZSA9IHRhcmdldC5uYW1lCiAgICAgICAgICAgICAgICBidWYgPSBpby5CeXRlc0lPKCkKICAgICAgICAgICAgICAgIG9yaWdfc2F2ZShvYmosIGJ1ZikKICAgICAgICAgICAgICAgIHBheWxvYWQgPSBidWYuZ2V0dmFsdWUoKQogICAgICAgICAgICAgICAgIyBPbmx5IHF1ZXVlIGludGVybWVkaWF0ZSBHXy9EXyByZXN1bWUgc3RhdGVzIGFzeW5jaHJvbm91c2x5LgogICAgICAgICAgICAgICAgIyBGaW5hbC9pbnRlcnZhbCBleHBvcnRlZCBpbmZlcmVuY2Ugd2VpZ2h0cyAoZS5nLiBNb2RlbF8yMDBlXzQwMDBzLnB0aCkKICAgICAgICAgICAgICAgICMgYXJlIHdyaXR0ZW4gc3luY2hyb25vdXNseSAmIGF0b21pY2FsbHkgc28gcHJvY2VzcyBleGl0IG5ldmVyIGRyb3BzIHRoZW0uCiAgICAgICAgICAgICAgICBpZiBmbmFtZS5zdGFydHN3aXRoKCgiR18iLCAiRF8iKSk6CiAgICAgICAgICAgICAgICAgICAgZnV0ID0gX1NBVkVfUE9PTC5zdWJtaXQoX3dyaXRlX2F0b21pYywgdGFyZ2V0LCBwYXlsb2FkKQogICAgICAgICAgICAgICAgICAgIHdpdGggX1NBVkVfTE9DSzoKICAgICAgICAgICAgICAgICAgICAgICAgX1BFTkRJTkdfU0FWRVNbc3RyKHRhcmdldCldID0gZnV0CiAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgIGZsdXNoX3NhdmVzKCkKICAgICAgICAgICAgICAgICAgICBfd3JpdGVfYXRvbWljKHRhcmdldCwgcGF5bG9hZCkKICAgICAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgcmV0dXJuIG9yaWdfc2F2ZShvYmosIGYsICphcmdzLCAqKmt3YXJncykKCiAgICBkZWYgc2FmZV9sb2FkKGY6IEFueSwgKmFyZ3M6IEFueSwgKiprd2FyZ3M6IEFueSkgLT4gQW55OgogICAgICAgIGlmIGlzaW5zdGFuY2UoZiwgKHN0ciwgb3MuUGF0aExpa2UpKToKICAgICAgICAgICAga2V5ID0gc3RyKFBhdGgob3MuZnNwYXRoKGYpKS5yZXNvbHZlKCkpCiAgICAgICAgICAgIHdpdGggX1NBVkVfTE9DSzoKICAgICAgICAgICAgICAgIGZ1dCA9IF9QRU5ESU5HX1NBVkVTLmdldChrZXkpCiAgICAgICAgICAgIGlmIGZ1dCBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIGZ1dC5yZXN1bHQoKQogICAgICAgIHJldHVybiBvcmlnX2xvYWQoZiwgKmFyZ3MsICoqa3dhcmdzKQoKICAgIGZhc3Rfc2F2ZS5fbm9yZWdyZXRzX3BhdGNoZWQgPSBUcnVlICAjIHR5cGU6IGlnbm9yZVthdHRyLWRlZmluZWRdCiAgICB0b3JjaC5zYXZlID0gZmFzdF9zYXZlICAjIHR5cGU6IGlnbm9yZVthc3NpZ25tZW50XQogICAgdG9yY2gubG9hZCA9IHNhZmVfbG9hZCAgIyB0eXBlOiBpZ25vcmVbYXNzaWdubWVudF0KCgpkZWYgYXBwbHkoKSAtPiBOb25lOgogICAgZ2xvYmFsIF9BUFBMSUVECiAgICBpZiBfQVBQTElFRCBvciBvcy5lbnZpcm9uLmdldCgiTlJfVFVSQk9fRU5BQkxFRCIsICIxIikgPT0gIjAiOgogICAgICAgIHJldHVybgogICAgX0FQUExJRUQgPSBUcnVlCiAgICBmb3Igc3RlcCBpbiAoCiAgICAgICAgX2FwcGx5X2N1ZGFfdHVuaW5nLAogICAgICAgIF9wYXRjaF9kYXRhbG9hZGVyLAogICAgICAgIF9wYXRjaF9vcHRpbWl6ZXIsCiAgICAgICAgX3BhdGNoX2RkcCwKICAgICAgICBfcGF0Y2hfdGVuc29yYm9hcmRfaW1hZ2VzLAogICAgICAgIF9wYXRjaF9yYW1fY2FjaGUsCiAgICAgICAgX3BhdGNoX2FzeW5jX3NhdmUsCiAgICApOgogICAgICAgIHRyeToKICAgICAgICAgICAgc3RlcCgpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgcGFzcwo="))
train_py = repo_dir / "rvc" / "train" / "train.py"
if train_py.is_file():
    _src = train_py.read_text(encoding="utf-8", errors="replace")
    if "noregrets_turbo.apply()" not in _src:
        _lines = _src.splitlines(keepends=True)
        _ins = 0
        for _i, _ln in enumerate(_lines[:40]):
            if _ln.startswith("from __future__ import"):
                _ins = _i + 1
        _hook = "import os as _nr_os, sys as _nr_sys\n_nr_sys.path.insert(0, _nr_os.getcwd())\ntry:\n    import noregrets_turbo\n    noregrets_turbo.apply()\nexcept Exception:\n    pass\n"
        _lines.insert(_ins, _hook)
        train_py.write_text("".join(_lines), encoding="utf-8")

# Seamless auto-update: fetch latest published WebUI files (or extract from NoRegretsRVC_Kaggle.ipynb) on GitHub main (keeps embedded build as fallback)
import json as _upd_json, re as _upd_re
_gh_updated_any = False
for _gh_rel, _dst_file, _valid_marker in (
    ("streamlit_app.py", repo_dir / "streamlit_app.py", "CRIMSON_CSS"),
    ("noregrets_turbo.py", repo_dir / "noregrets_turbo.py", "def apply("),
    (".streamlit/config.toml", repo_dir / ".streamlit" / "config.toml", "primaryColor"),
):
    try:
        _req_u = urllib.request.Request(f"https://raw.githubusercontent.com/HunterThatsMe/NoRegretsRVC/main/{_gh_rel}", headers={"User-Agent": "NoRegretsRVC"})
        with urllib.request.urlopen(_req_u, timeout=8) as _resp_u:
            _raw_u = _resp_u.read()
        if len(_raw_u) > 64 and _valid_marker in _raw_u.decode("utf-8", errors="replace"):
            if _dst_file.name != "noregrets_turbo.py" or True:
                _dst_file.write_bytes(_raw_u)
                _gh_updated_any = True
    except Exception:
        pass
if not _gh_updated_any:
    for _nb_rel in ("NoRegretsRVC_Kaggle.ipynb", "assets/NoRegretsRVC_Kaggle.ipynb"):
        try:
            _req_nb = urllib.request.Request(f"https://raw.githubusercontent.com/HunterThatsMe/NoRegretsRVC/main/{_nb_rel}", headers={"User-Agent": "NoRegretsRVC"})
            with urllib.request.urlopen(_req_nb, timeout=10) as _resp_nb:
                _nb_obj = _upd_json.loads(_resp_nb.read().decode("utf-8", errors="replace"))
            _cell_src = "\n".join("".join(c.get("source", [])) if isinstance(c.get("source"), list) else str(c.get("source", "")) for c in _nb_obj.get("cells", []) if c.get("cell_type") == "code")
            for _pat_rel, _dst_file, _valid_marker in (
                (r'\(repo_dir / "streamlit_app\.py"\)\.write_bytes\(base64\.b64decode\("([A-Za-z0-9+/=]+)"\)\)', repo_dir / "streamlit_app.py", "CRIMSON_CSS"),
                (r'\(repo_dir / "noregrets_turbo\.py"\)\.write_bytes\(base64\.b64decode\("([A-Za-z0-9+/=]+)"\)\)', repo_dir / "noregrets_turbo.py", "def apply("),
                (r'\(repo_dir / "\.streamlit" / "config\.toml"\)\.write_bytes\(base64\.b64decode\("([A-Za-z0-9+/=]+)"\)\)', repo_dir / ".streamlit" / "config.toml", "primaryColor"),
            ):
                _bm = _upd_re.search(_pat_rel, _cell_src)
                if _bm:
                    _dec = base64.b64decode(_bm.group(1))
                    if len(_dec) > 64 and _valid_marker in _dec.decode("utf-8", errors="replace"):
                        if _dst_file.name != "noregrets_turbo.py" or True:
                            _dst_file.write_bytes(_dec)
                            _gh_updated_any = True
            if _gh_updated_any:
                break
        except Exception:
            pass

# Best-effort prefetch of the 6 Free Samples WAVs from https://github.com/HunterThatsMe/NoRegretsRVC
fs_dir = repo_dir / "assets" / "free_samples"
au_dir = repo_dir / "assets" / "audios"
fs_dir.mkdir(parents=True, exist_ok=True)
au_dir.mkdir(parents=True, exist_ok=True)
for _wav_name in (
    "MaleSpeakingTest-1_931b5ffd.wav",
    "MaleSpeakingTest-2_c2e6e7fe.wav",
    "MaleSpeakingTest-3_d78e8233.wav",
    "FemaleSpeakingTest-1_bb3d674f.wav",
    "FemaleSpeakingTest-2_d1823132.wav",
    "FemaleSpeakingTest-3_458bd4fc.wav",
):
    _dst_fs = fs_dir / _wav_name
    if not _dst_fs.is_file():
        try:
            urllib.request.urlretrieve(f"https://raw.githubusercontent.com/HunterThatsMe/NoRegretsRVC/main/public/assets/aistudio/{_wav_name}", _dst_fs)
            shutil.copy2(_dst_fs, au_dir / _wav_name)
        except Exception:
            pass

os.chdir(repo_dir)
prereq_cmd = [py_exec, "core.py", "prerequisites", "--models", "--exe", "--pretraineds-hifigan"]
subprocess.run(prereq_cmd, check=True)

# Pre-boot Model & Custom Pretrains Downloader (HF repo auto-discovery, ZIP & G/D .pth)
import json as _json
import urllib.parse
import zipfile as _zipfile
boot_downloads = [
    {
        "name": "RMVPE Pitch Model (Default)",
        "url": "https://huggingface.co/IAHispano/Applio/resolve/main/Resources/predictors/rmvpe.pt",
        "category": "pretrained"
    },
    {
        "name": "ContentVec Embedder",
        "url": "https://huggingface.co/IAHispano/Applio/resolve/main/Resources/embedders/contentvec/pytorch_model.bin",
        "category": "embedder"
    }
]
dl_dir = repo_dir / "logs" / "downloads"
custom_pretrain_dir = repo_dir / "rvc" / "models" / "pretraineds" / "custom"
dl_dir.mkdir(parents=True, exist_ok=True)
custom_pretrain_dir.mkdir(parents=True, exist_ok=True)
def _download_hf_repo_pretrains(_repo_url, _dst_dir):
    _parts = [p for p in urllib.parse.urlsplit(_repo_url).path.strip("/").split("/") if p]
    if len(_parts) < 2:
        return
    _owner_repo = f"{_parts[0]}/{_parts[1]}"
    _api = f"https://huggingface.co/api/models/{_owner_repo}/tree/main"
    with urllib.request.urlopen(_api, timeout=30) as _resp:
        _tree = _json.loads(_resp.read().decode("utf-8"))
    for _node in _tree:
        _fpath = _node.get("path", "")
        if _fpath.lower().endswith(".pth"):
            _fname = Path(_fpath).name
            _out = _dst_dir / _fname
            if not _out.is_file():
                _file_url = f"https://huggingface.co/{_owner_repo}/resolve/main/{urllib.parse.quote(_fpath)}?download=true"
                print("Downloading discovered pretrain", _fname, "->", _out)
                urllib.request.urlretrieve(_file_url, _out)
for entry in boot_downloads:
    _url = entry["url"].strip()
    _cat = entry.get("category", "voice")
    _target_dir = custom_pretrain_dir if _cat == "pretrained" else dl_dir
    _parsed_path = urllib.parse.urlsplit(_url).path
    _raw_name = urllib.parse.unquote(_parsed_path.split("/")[-1]) or entry["name"]
    try:
        if _cat == "pretrained" and "huggingface.co/" in _url.lower() and "/resolve/" not in _url and not _parsed_path.lower().endswith((".pth", ".zip")):
            _download_hf_repo_pretrains(_url, _target_dir)
        elif _cat == "pretrained" and _parsed_path.lower().endswith(".zip"):
            _zip_tmp = _target_dir / (_raw_name if _raw_name.endswith(".zip") else "custom_pretrains.zip")
            print("Downloading pretrained ZIP", _url, "->", _zip_tmp)
            urllib.request.urlretrieve(_url, _zip_tmp)
            with _zipfile.ZipFile(_zip_tmp, "r") as _zf:
                for _member in _zf.namelist():
                    if not _member.endswith("/") and Path(_member).name and _member.lower().endswith(".pth"):
                        _out_pth = _target_dir / Path(_member).name
                        _out_pth.write_bytes(_zf.read(_member))
                        print("Extracted", _out_pth.name)
            _zip_tmp.unlink(missing_ok=True)
        else:
            _dst = _target_dir / _raw_name
            if not _dst.is_file():
                print("Downloading", entry["name"], "->", _dst)
                urllib.request.urlretrieve(_url, _dst)
    except Exception as _dl_err:
        print("Skipping unreachable download", _url, ":", _dl_err)

cloudflared_path = repo_dir.parent / "cloudflared"
cloudflared_url = "https://github.com/cloudflare/cloudflared/releases/download/2026.9.0/cloudflared-linux-amd64"
cloudflared_sha256 = "53b7a7a5420d188758d24341294acb0d1bca54296548ac05e38811a694ac6134"
if not (cloudflared_path.is_file() and hashlib.sha256(cloudflared_path.read_bytes()).hexdigest() == cloudflared_sha256):
    tmp = cloudflared_path.with_suffix(".download")
    with urllib.request.urlopen(cloudflared_url, timeout=120) as resp, open(tmp, "wb") as dst:
        shutil.copyfileobj(resp, dst)
    if hashlib.sha256(tmp.read_bytes()).hexdigest() != cloudflared_sha256:
        tmp.unlink(missing_ok=True)
        raise RuntimeError("cloudflared checksum verification failed.")
    os.replace(tmp, cloudflared_path)
cloudflared_path.chmod(0o755)
clear_output()
print("NoRegretsRVC Streamlit setup complete! Model: MyVoice (32000 · HiFi-GAN)")


## 2. Launch Streamlit WebUI & Live Monitor


In [ ]:
import json
import os
import re
import secrets
import shutil
import subprocess
import sys
import time
from collections import deque
from pathlib import Path
from IPython.display import clear_output

Tunnel = "Gradio + Cloudflare" #@param ["Gradio + Cloudflare", "Horizon"]
horizon_id = "" #@param {type:"string"}

repo_dir = Path("/kaggle/working/program_ml")
cloudflared_path = repo_dir.parent / "cloudflared"
py_exec = "/kaggle/.venv/bin/python"
if not os.path.exists(py_exec):
    py_exec = sys.executable
for _st_dir in (repo_dir / ".streamlit", Path.home() / ".streamlit"):
    _st_dir.mkdir(parents=True, exist_ok=True)
    (_st_dir / "credentials.toml").write_text('[general]\nemail = ""\n', encoding="utf-8")

app_user = "noregrets"
app_pass = secrets.token_urlsafe(24)

cf_pattern = r"https://[a-zA-Z0-9-]+\.trycloudflare\.com"
hz_pattern = r"https://[a-zA-Z0-9.-]+\.hrzn\.run"
tunnel_procs = []
tunnel_logs = []
app_proc = None

def start_tunnel(cmd, name, pattern):
    out_path = repo_dir / f"{name}.log"
    for _attempt in range(2):
        f = open(out_path, "w", encoding="utf-8")
        proc = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT, text=True)
        tunnel_procs.append(proc)
        tunnel_logs.append(f)
        deadline = time.time() + 90
        while time.time() < deadline:
            txt = out_path.read_text(encoding="utf-8", errors="replace") if out_path.exists() else ""
            m = re.search(pattern, txt)
            if m:
                return m.group(0)
            if proc.poll() is not None:
                break
            time.sleep(1)
        if proc.poll() is None:
            proc.terminate()
        time.sleep(1.5)
    raise RuntimeError(f"Tunnel {name} failed to emit URL")

os.chdir(repo_dir)
if shutil.which("fuser"):
    subprocess.run(["fuser", "-k", "6969/tcp"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

if Tunnel == "Horizon":
    if not horizon_id.strip():
        raise ValueError("Enter your horizon_id before running Horizon tunnel.")
    subprocess.run(["npm", "install", "-g", "@hrzn/cli"], check=True)
    subprocess.run(["hrzn", "login", horizon_id], check=True)

# --- NoRegretsRVC Kaggle Live Monitor (Red title, 🟥 progress bar, (saved) markers, session separators) ---
BOLD = "\033[1m"
RED_BOLD = "\033[1;31m"
GREEN_BOLD = "\033[1;32m"
RESET = "\033[0m"
ANSI_RE = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")
EPOCH_RE = re.compile(r"epoch=(\d+)\s*\|\s*step=(\d+).*?training_speed=(\d+:\d{2}:\d{2})(?:.*?lowest_value=([0-9.]+))?")
TQDM_RE = re.compile(r"(\d+)%\|[^|]*\|\s*(\d+)/(\d+)\s*\[([^,\]]+),\s*([0-9.]+\s*(?:it/s|s/it))")
LOCAL_URL_RE = re.compile(r"(?:Running on local URL|Local URL|Network URL):\s*(https?://\S+)")
NOISE_RE = re.compile(r"(FutureWarning|UserWarning|DeprecationWarning|RuntimeWarning|pkg_resources|warnings\.warn|oneDNN|tensorflow/core|computation_placer|numa_node|Successfully opened dynamic library|weights_only|HTTP Request:|Checking out|libpng warning|ALSA|jack server|You can now view your Streamlit|local_sources_watcher|No module named 'torchvision'|get_module_paths)", re.IGNORECASE)
MILESTONE_RE = re.compile(r"(Starting (?:preprocess|extract|index|training)|⏳ Starting|⏳ Downloading pretrained|✔ .* complete!|✔ Pretrained|Downloaded .* from HF|Downloaded [GD] weight|Extracted .*|✖ .* failed|Preprocess completed|preprocessed successfully|Starting pitch extraction|Pitch extraction completed|Starting embedding extraction|Embedding extraction completed|Saved index file|Saving checkpoint|Saving model|Saving weights|Index file.*generated|Using .* vocoder|Using .* for training)", re.IGNORECASE)
ERROR_START_RE = re.compile(r"(Traceback \(most recent call last\)|NameError:|TypeError:|KeyError:|AttributeError:|SyntaxError:|RuntimeError:|ValueError:|FileNotFoundError:|OSError:|ModuleNotFoundError:|ImportError:|CUDA out of memory|OutOfMemoryError|Exception:|✖ .* failed|\bERROR:)")
SAVE_LOG_RE = re.compile(r"(Saving checkpoint|Saving model|Saving weights|Saved checkpoint|Saved model)", re.IGNORECASE)

session_links = []
milestones = deque(maxlen=6)
errors = deque(maxlen=25)
train_sessions = []
active_training = False
save_pending = False
seen_epochs = set()
live_mid_epoch = ""
last_tq_render = 0.0
tb_lines_seen = 0
last_completed_epoch = 0
active_meta_path = repo_dir / "logs" / ".nr_active_train.json"

def read_train_targets():
    tot_ep = 200
    save_ev = 5
    if active_meta_path.is_file():
        try:
            md = json.loads(active_meta_path.read_text(encoding="utf-8"))
            tot_ep = int(md.get("total_epoch") or tot_ep)
            save_ev = int(md.get("save_every_epoch") or save_ev)
            m_slug = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(md.get("model_name") or "").strip())
            if m_slug:
                args_f = repo_dir / "logs" / m_slug / "nr_train.args.json"
                if args_f.is_file():
                    ad = json.loads(args_f.read_text(encoding="utf-8"))
                    tot_ep = int(ad.get("total_epoch") or tot_ep)
                    save_ev = int(ad.get("save_every_epoch") or save_ev)
        except Exception:
            pass
    return max(1, tot_ep), max(1, save_ev)

def ensure_open_session(resumed=False):
    global seen_epochs
    if not train_sessions or train_sessions[-1]["footer"] != "":
        was_stopped = bool(train_sessions and train_sessions[-1]["footer"] == "Training Stopped")
        train_sessions.append({
            "header_note": "Training Resumed" if (resumed or was_stopped) else "",
            "rows": deque(maxlen=15),
            "footer": "",
        })
        seen_epochs = set()

def upsert_link(label, value):
    for idx, (k, _) in enumerate(session_links):
        if k.strip() == label.strip():
            session_links[idx] = (label, value)
            return
    session_links.append((label, value))

def render_monitor(status="Running"):
    all_rows = [r for s in train_sessions for r in s["rows"]]
    w_speed = max(9, max((len(r[0]) for r in all_rows), default=0))
    w_step = max(8, max((len(r[1]) for r in all_rows), default=0))
    w_epoch = max(14, max((len(r[2]) for r in all_rows), default=0))
    tot_ep, _ = read_train_targets()
    cur_ep_n = min(tot_ep, last_completed_epoch)
    pct_ep = (cur_ep_n / max(1, tot_ep)) * 100.0
    filled_b = max(0, min(16, int(round((cur_ep_n / max(1, tot_ep)) * 16))))
    prog_bar = ("🟥" * filled_b) + ("⬛" * (16 - filled_b))
    lines = [
        f"{RED_BOLD}NoRegretsRVC Live Monitor (Beta 0.0.0){RESET}  [{status}]",
        "=" * 64,
        "",
    ]
    if app_pass:
        lines.append(f"WebUI Login      : {app_user} / {app_pass}")
        lines.append("")
    for label, val in session_links:
        if val:
            lines.append(f"{label:<17}: {val}")
            lines.append("")
    lines.append("-" * 64)
    lines.append("")
    if milestones:
        for m in milestones:
            lines.append(f"[Stage] {m}")
        lines.append("")
    if active_training or train_sessions:
        lines.append(f"{BOLD}Progress         :{RESET} {prog_bar}  {cur_ep_n}/{tot_ep} ({pct_ep:.1f}%)")
        lines.append("")
        live_txt = live_mid_epoch if live_mid_epoch else ("Idle — waiting for next epoch batches..." if active_training else "Session idle.")
        lines.append(f"{BOLD}Live Epoch:{RESET} {live_txt}")
        lines.append("")
    if train_sessions:
        for sess in train_sessions[-3:]:
            if sess["header_note"]:
                lines.append(f"{BOLD}{sess['header_note']}{RESET}")
                lines.append("")
                lines.append("")
            lines.append(f"{'Speed':<{w_speed}} | {'Step':<{w_step}} | {BOLD}{'Epoch':<{w_epoch}}{RESET} | Lowest Loss")
            lines.append(f"{'-' * w_speed}-+-{'-' * w_step}-+-{'-' * w_epoch}-+------------")
            if sess["rows"]:
                for spd, stp, ep, low in sess["rows"]:
                    lines.append(f"{spd:<{w_speed}} | {stp:<{w_step}} | {BOLD}{ep:<{w_epoch}}{RESET} | {low}")
            else:
                lines.append("Waiting for first completed epoch...")
            if sess["footer"]:
                lines.append("")
                lines.append("")
                f_col = RED_BOLD if sess["footer"] == "Training Stopped" else GREEN_BOLD
                lines.append(f"{f_col}{sess['footer']}{RESET}")
                lines.append("")
    elif not errors:
        lines.append("Waiting for training epochs...")
    if errors:
        lines.append("")
        lines.append(f"{BOLD}Errors:{RESET}")
        for err in errors:
            lines.append(f"  {err}")
    clear_output(wait=True)
    print("\n".join(lines), flush=True)

try:
    env = os.environ.copy()
    if app_pass:
        env["APPLIO_AUTH_USERNAME"] = app_user
        env["APPLIO_AUTH_PASSWORD"] = app_pass
    env.update({
        "PYTHONWARNINGS": "ignore",
        "PYTHONUNBUFFERED": "1",
        "TF_CPP_MIN_LOG_LEVEL": "3",
        "TORCH_CPP_LOG_LEVEL": "ERROR",
        "TOKENIZERS_PARALLELISM": "false",
        "STREAMLIT_SERVER_FILE_WATCHER_TYPE": "none",
        "STREAMLIT_SERVER_RUN_ON_SAVE": "false",
        "STREAMLIT_BROWSER_GATHER_USAGE_STATS": "false",
    })

    if Tunnel == "Gradio + Cloudflare":
        cf_url = start_tunnel([str(cloudflared_path), "tunnel", "--no-autoupdate", "--protocol", "http2", "--url", "http://127.0.0.1:6969"], "webui-cf", cf_pattern)
        upsert_link("NoRegretsRVC URL", cf_url)
        upsert_link("Cloudflare URL", cf_url)
    else:
        web_url = start_tunnel(["hrzn", "tunnel", "http://localhost:6969"], "webui-hrzn", hz_pattern)
        upsert_link("NoRegretsRVC URL", web_url)

    app_cmd = [
        py_exec, "-u", "-m", "streamlit", "run", "streamlit_app.py",
        "--server.address", "0.0.0.0",
        "--server.port", "6969",
        "--server.headless", "true",
        "--server.fileWatcherType", "none",
        "--server.runOnSave", "false",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
    ]
    restart_flag_p = repo_dir / "logs" / ".nr_restart_webui"
    while True:
        render_monitor("Starting NoRegretsRVC WebUI")
        app_proc = subprocess.Popen(app_cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1)
        assert app_proc.stdout is not None
        for raw_line in app_proc.stdout:
            line = ANSI_RE.sub("", raw_line).replace("\r", "").strip()
            if not line:
                continue
            loc_m = LOCAL_URL_RE.search(line)
            if loc_m:
                upsert_link("Local UI URL", loc_m.group(1))
                render_monitor("Ready")
                continue
            if "⏳ Starting training" in line:
                ensure_open_session()
                active_training = True
                save_pending = False
                live_mid_epoch = ""
                if not milestones or milestones[-1] != line:
                    milestones.append(line)
                render_monitor("Training")
                continue
            if "Stop signal sent" in line or "✖ Training exited with error code" in line:
                if train_sessions and train_sessions[-1]["footer"] == "":
                    train_sessions[-1]["footer"] = "Training Stopped"
                active_training = False
                live_mid_epoch = ""
                render_monitor("Stopped")
                continue
            if "✔ Training complete!" in line or "Training finished" in line or "Training completed" in line:
                if train_sessions and train_sessions[-1]["footer"] == "":
                    train_sessions[-1]["footer"] = "Training Complete"
                active_training = False
                live_mid_epoch = ""
                if not milestones or milestones[-1] != line:
                    milestones.append(line)
                render_monitor("Complete")
                continue
            if SAVE_LOG_RE.search(line):
                save_pending = True
                if not milestones or milestones[-1] != line:
                    milestones.append(line)
                    render_monitor("Training" if active_training else "Running")
                continue
            ep_m = EPOCH_RE.search(line)
            if ep_m:
                ensure_open_session()
                active_training = True
                ep_num, stp_num, spd_str = ep_m.group(1), ep_m.group(2), ep_m.group(3)
                low_str = ep_m.group(4) or "-"
                ep_int = int(ep_num)
                last_completed_epoch = max(last_completed_epoch, ep_int)
                tot_ep_cur, save_ev_cur = read_train_targets()
                is_saved = save_pending or (ep_int % save_ev_cur == 0) or (ep_int == tot_ep_cur)
                save_pending = False
                ep_lbl = f"{ep_num} (saved)" if is_saved else ep_num
                key = (ep_num, stp_num)
                if key not in seen_epochs:
                    seen_epochs.add(key)
                    train_sessions[-1]["rows"].append((spd_str, stp_num, ep_lbl, low_str))
                    live_mid_epoch = ""
                    render_monitor("Training")
                continue
            tq_m = TQDM_RE.search(line)
            if tq_m:
                now_ts = time.time()
                if now_ts - last_tq_render >= 5.0 or tq_m.group(1) == "100":
                    last_tq_render = now_ts
                    live_mid_epoch = f"{tq_m.group(1)}% ({tq_m.group(2)}/{tq_m.group(3)}) · {tq_m.group(4)} · {tq_m.group(5)}"
                    render_monitor("Training" if active_training else "Running")
                continue
            if NOISE_RE.search(line) or "%|" in line or "it/s]" in line or "s/it]" in line:
                continue
            if tb_lines_seen > 0:
                if not errors or errors[-1] != line:
                    errors.append(line)
                tb_lines_seen = 0 if re.match(r"^[A-Za-z_]*(Error|Exception):", line) else (tb_lines_seen - 1)
                render_monitor("Error")
                continue
            if ERROR_START_RE.search(line):
                if not errors or errors[-1] != line:
                    errors.append(line)
                if "Traceback" in line:
                    tb_lines_seen = 25
                render_monitor("Error")
                continue
            if MILESTONE_RE.search(line):
                if not milestones or milestones[-1] != line:
                    milestones.append(line)
                    render_monitor("Running")
        app_proc.wait()
        if restart_flag_p.is_file():
            restart_flag_p.unlink(missing_ok=True)
            if shutil.which("fuser"):
                subprocess.run(["fuser", "-k", "6969/tcp"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            continue
        break
finally:
    for p in reversed(tunnel_procs):
        if p.poll() is None: p.terminate()
    if app_proc is not None and app_proc.poll() is None:
        app_proc.terminate()
    for f in tunnel_logs:
        f.close()
    if shutil.which("fuser"):
        subprocess.run(["fuser", "-k", "6969/tcp"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
